# Proyecto ML End-to-End: Predicción de Riesgo Crediticio
**Objetivo:** Clasificar clientes según su riesgo de impago utilizando el dataset 'German Credit Data'.

## 1. Definición del Problema
* **Clase 0:** Buen pagador (Bajo riesgo).
* **Clase 1:** Mal pagador (Alto riesgo / Default)..
* **Métrica objetivo:** Maximizar el **Recall (Clase 1)** para reducir pérdidas financieras por falsos negativos.

**Estrategia de Negocio:** El error más costoso es el **Falso Negativo** (aprobar a un cliente que resultará ser un mal pagador), ya que implica una pérdida directa de capital. Por ello, la estrategia de este modelo se centra en maximizar el **Recall de la clase 1** y el **ROC-AUC**, asegurando que la institución identifique a la mayor cantidad posible de clientes de riesgo.

![recall_clase_1][def]

[def]: ../reports/figures/business_strategy.png

## 1.a. Ciclo de Vida del Proyecto (Pipeline End-to-End)
El proyecto sigue una estructura profesional de 10 pasos:



1. Ingesta: Carga reproducible y validación de columnas.


2. Limpieza: Manejo de tipos y nulos.


3. EDA: Análisis de señales predictivas y outliers.


4. Feature Engineering: Creación de variables y transformaciones.


5. Pipeline ML: Integración de preprocesamiento y modelo.


6. Entrenamiento: Validación cruzada estratificada.


7. Evaluación: Análisis de matriz de confusión y métricas de negocio.


8. Serialización: Guardado del pipeline completo con joblib.


9. Testing: Pruebas de regresión de datos y código.
10. Deployment: Despliegue mediante API y monitoreo de Data Drift



## 2. Entendimiento y Limpieza de Datos
Se utilizó el dataset German Credit Data. Durante la fase de exploración y limpieza (EDA), se realizaron las siguientes acciones técnicas:

- **Diagnóstico de Clases:** El dataset presenta un desbalance moderado (70% buenos / 30% malos), lo que requiere técnicas de compensación en el modelo.

- T**ratamiento de Nulos:** En variables como cuentas bancarias, la ausencia de datos se categorizó como "unknown", ya que el no tener historial bancario registrado es un factor predictivo relevante.

- **Análisis de Distribución:** Se detectó una asimetría fuerte (skewness) en variables como Credit amount, lo que justificó el uso de transformaciones logarítmicas para normalizar los datos.

## 3. Estrategia de Preprocesamiento y Feature Engineering
Se aplicó el siguiente plan de transformación por tipo de variable:

| Variable(s)                     | Técnica Aplicada        | Justificación                                                                 |
|----------------------------------|------------------------|-------------------------------------------------------------------------------|
| Credit amount / Duration         | log1p + RobustScaler   | Reduce la asimetría y mitiga el impacto de valores extremos (outliers).      |
| Age                              | RobustScaler           | Escalamiento resistente a outliers sin distorsionar la distribución de edades. |
| Saving / Checking accounts       | Ordinal Encoding       | Respeta el orden económico natural (ej. little < moderate < rich).           |
| Sex / Housing / Purpose          | One-Hot Encoding       | Trata categorías nominales sin imponer una jerarquía artificial.             |

**Variables Derivadas:**

- Credit_per_Month (Monto/Duración) para reflejar la carga financiera mensual.

## 4. Modelado y Evaluación (Iteración 1)
Se utilizó un ExtraTreesClassifier con pesos de clase balanceados.

**Resultados:**

**ROC-AUC:** 0.70.

**Recall clase 1 (Bad):** 0.32 (Solo se detecta el 32% de los riesgos).

**Diagnóstico:** El modelo es demasiado conservador y sesgado hacia la clase "buena". Es inaceptable para un escenario real.

![recall_clase_1][def]

[def]: ../reports/figures/new_features.png

**Variables Derivadas (Nuevas):**

- **Credit_per_Month:** Relación entre el monto y la duración para reflejar la carga financiera mensual.

- **Age_Duration_Ratio:** Captura el perfil de riesgo temporal del solicitante.


## 4.a. Detalles de los experimentos (iteración 1)
En esta fase inicial, se estableció un Baseline (punto de partida) para medir la capacidad predictiva básica del sistema antes de aplicar optimizaciones avanzadas.


- **Modelo Utilizado:** ExtraTreesClassifier con 400 estimadores.


- **Configuración de Clase:** Se aplicó class_weight="balanced" para compensar el desbalance natural de los datos (70% buenos / 30% malos).


- **Validación:** Se utilizó un esquema de partición Stratified Train/Test Split (80/20) para asegurar que la proporción de clientes "malos" fuera la misma en ambos conjuntos.

**Resultados Obtenidos (Métricas de MLflow):**

| Métrica | Resultado | Interpretación |
| :--- | :--- | :--- |
| **Accuracy** | 0.74 | Precisión global aceptable, pero engañosa debido al desbalance. |
| **ROC-AUC** | 0.70 | Capacidad predictiva moderada; se busca superar el 0.75 en la siguiente fase. |
| **Recall (Clase 1 - Bad)** | 0.32 | **Punto crítico:** Solo se detecta el 32% de los clientes riesgosos. |
| **Recall (Clase 0 - Good)** | 0.92 | Excelente detección de clientes seguros. |

## 5. Optimización del Umbral de Decisión (Threshold Tuning)
En la **Iteración 1**, el modelo utilizó el umbral por defecto de **0.5**. Esto significa que solo si la probabilidad de impago es mayor al 50%, el cliente es clasificado como "Riesgo".

Sin embargo, en el negocio bancario, el costo de un **Falso Negativo** (aprobar a alguien que no paga) es mucho mayor que el de un Falso Positivo (rechazar a alguien que sí pagaría). Por ello, es necesario reducir el umbral.

| Umbral (Threshold) | Impacto en Recall (Clase 1) | Impacto en Precision | Consecuencia para el Negocio |
| :--- | :--- | :--- | :--- |
| **0.50 (Base)** | 0.32 (Bajo) | Alto | Se aprueban muchos créditos, pero la pérdida por morosidad es muy alta. |
| **0.35 (Recomendado)** | **0.65 (Medio/Alto)** | Moderado | **Punto de equilibrio:** Se detectan el doble de morosos sacrificando algunos clientes buenos. |
| **0.20 (Conservador)** | 0.85 (Muy Alto) | Bajo | Se minimiza el riesgo casi a cero, pero se rechazan demasiadas solicitudes legítimas. |

![recall_clase_1][def]

[def]: ../reports/figures/optimization.png

## Justificación Técnica del Cambio
Al bajar el umbral a 0.35:

1. **Aumento del Recall:** Forzamos al modelo a ser más sensible ante cualquier "bandera roja" financiera.

2. **Reducción de Pérdidas:** Aunque el Accuracy total pueda bajar ligeramente, el beneficio económico de evitar impagos compensa con creces la pérdida de algunos intereses por rechazos preventivos.

3. **Curva de Costos:** Se busca el punto donde el costo de adquisición de clientes y el costo de capital perdido se minimizan simultáneamente.